# Documentation de la classe `RiskIndicators`

La classe `RiskIndicators` (`kadi.weather.risk.RiskIndicators`) est dédiée à l'évaluation des risques climatiques, notamment les **indicateurs de sécheresse** et les **probabilités de précipitations à court terme**.

## 1. Initialisation
L'analyse des risques s'appuie à la fois sur l'historique (pour détecter les déficits) et sur les prévisions (pour évaluer le risque immédiat). L'initialisation requiert donc :
- L'instance `Location`
- L'historique des précipitations (`pandas.Series`)
- Le DataFrame de prévisions (`forecast_data`)

In [1]:
# Changer de répertoire
import os
from pathlib import Path
print(Path.cwd())
new_dir = Path("~/Bureau/kadipy/").expanduser().resolve()
os.chdir(new_dir)
print(Path.cwd())

from kadi.weather.location import Location
from kadi.weather.data import WeatherData
from kadi.weather.risk import RiskIndicators

# Récupération des données nécessaires
loc = Location(latitude=9.3333, longitude=2.6333, name='Parakou')
weather = WeatherData(loc)
hist = weather.fetch_historical(months_back=6)
forecast = weather.fetch_forecast(days=7)

risk = RiskIndicators(loc, hist['precipitation'], forecast)
print("Indicateurs de risques initialisés.")

/home/dels/Bureau/kadipy/docs/weather
/home/dels/Bureau/kadipy
Indicateurs de risques initialisés.


## 2. Indice de Sécheresse (`drought_index`)
**Méthode :** `drought_index(method: str = 'spi', window_months: int = 3) -> dict`

La classe permet de calculer trois grands types d'indicateurs de sécheresse :
- **SPI (Standardized Precipitation Index)** : C'est l'indice standard de l'OMM. Il compare le cumul des précipitations d'une fenêtre donnée (ex: 3 mois) par rapport à la moyenne historique. Un Z-Score négatif signale une sécheresse (`<-1.5` pour sévère).
- **Markov Transition** : Calcule les probabilités de transition (ex: probabilité qu'un jour sec soit suivi d'un autre jour sec `p_dry_dry`).
- **Hurst Exponent** : Évalue la "mémoire" ou persistance du climat. Un exposant > 0.5 indique qu'une période de sécheresse a tendance à se prolonger.

La méthode peut utiliser `method='combined'` pour tous les calculer.

In [6]:
# Calcul de l'indice combiné sur les 3 derniers mois
drought_info = risk.drought_index(method='combined', window_months=3)

print("--- Indicateurs de Sécheresse ---")
print(f"SPI (3 mois) : {drought_info['spi_3month']} -> Sévérité : {drought_info['drought_severity']}")
print(f"Probabilité de maintien d'une période sèche (Markov) : {drought_info['markov_p_dry'] * 100}%")
print(f"Exposant de Hurst : {drought_info['hurst_exponent']}")

--- Indicateurs de Sécheresse ---
SPI (3 mois) : 2.27 -> Sévérité : no_drought
Probabilité de maintien d'une période sèche (Markov) : 85.0%
Exposant de Hurst : 0.82


## 3. Probabilité de Pluie et Recommandations (`rain_probability`)
**Méthode :** `rain_probability(days_ahead: int = 1, min_rainfall_mm: float = 1.0) -> dict`

Évalue le risque de précipitation sur les jours à venir à partir du volume brut fourni par l'API Open-Meteo.
La méthode retourne non seulement les probabilités pour les jours futurs, mais ajoute également une **recommandation agronomique**. Par exemple, si la probabilité de forte pluie est élevée (>70%), il sera recommandé de repousser les traitements phytosanitaires pour éviter le lessivage.

In [4]:
# Probabilités de pluies significatives (>1mm) pour les 3 prochains jours
rain_risk = risk.rain_probability(days_ahead=3, min_rainfall_mm=1.0)

print("--- Risque Pluvieux ---")
print(f"Message du système : {rain_risk['message']}")
print(f"Recommandation     : {rain_risk['recommendation']}\n")

print("Détail :")
for key, value in rain_risk.items():
    if key not in ['message', 'recommendation']:
        print(f" - {key} : {value*100}%")

--- Risque Pluvieux ---
Message du système : 100% de chance de pluie dans les 3 jours.
Recommandation     : Risque de lessivage. Repousser les traitements phytosanitaires.

Détail :
 - tomorrow : 100.0%
 - 2_days : 100.0%
 - 3_days : 100.0%
